In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain accuracy                    0.648600                    0.515800   
             precision                   0.565210                    0.415629   
             recall                      0.696150                    0.524952   
             f1                          0.622876                    0.462630   
             kappa                       0.300298                    0.033679   
             MCC                         0.306780                    0.034535   
outputsTest  accuracy                    0.646700                    0.505700   
             precision                   0.558363                    0.407750   
             recall                      0.693884                    0.503963   
             f1                          0.617664                    0.449693   
             kappa                       0.296754                    0.010394   
             MCC                         0.303609                    0.010577   
outputsVal   accuracy                    0.658300                    0.507700   
             precision                   0.573412                    0.414605   
             recall                      0.706357                    0.517171   
             f1                          0.631845                    0.459015   
             kappa                       0.319618                    0.017661   
             MCC                         0.326759                    0.018375   

                        situation-dependent_vs_explicit  \
outputsTrain accuracy                          0.602600   
             precision                         0.670166   
             recall                            0.611458   
             f1                                0.638285   
             kappa                             0.199413   
             MCC                               0.201341   
outputsTest  accuracy                          0.597400   
             precision                         0.662399   
             recall                            0.613322   
             f1                                0.635673   
             kappa                             0.186249   
             MCC                               0.187727   
outputsVal   accuracy                          0.585400   
             precision                         0.658671   
             recall                            0.600167   
             f1                                0.627270   
             kappa                             0.161929   
             MCC                               0.163249   

                        non-persuasive_vs_persuasive  \
outputsTrain accuracy                       0.570700   
             precision                      0.424798   
             recall                         0.610304   
             f1                             0.499507   
             kappa                          0.144749   
             MCC                            0.152956   
outputsTest  accuracy                       0.556600   
             precision                      0.418999   
             recall                         0.586673   
             f1                             0.487598   
             kappa                          0.116408   
             MCC                            0.121806   
outputsVal   accuracy                       0.544700   
             precision                      0.393640   
             recall                         0.572766   
             f1                             0.464859   
             kappa                          0.092956   
             MCC                            0.098631   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTrain accuracy                   0.533800                  0.452200  
             precision                  0.258980                  0.131479  
             recall                     0.601747                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.312382,0.021162,0.184105,0.124464,0.103246,-0.127339
accuracy,0.651200,0.509733,0.595133,0.557333,0.540133,0.455733
f1,0.624128,0.457113,0.633743,0.483988,0.363797,0.193298
kappa,0.305557,0.020578,0.182530,0.118038,0.084873,-0.100023
precision,0.565662,0.412661,0.663745,0.412479,0.262141,0.133360
recall,0.698797,0.515362,0.608316,0.589915,0.602431,0.358110


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain pearson                    0.362718                    0.044178   
             spearman                   0.363069                    0.069722   
             MSE                        1.274564                    1.911643   
             RMSE                       1.127160                    1.380925   
             MAE                        0.903980                    1.060430   
outputsTest  pearson                    0.349905                    0.014208   
             spearman                   0.347958                    0.047141   
             MSE                        1.300191                    1.971584   
             RMSE                       1.138593                    1.402200   
             MAE                        0.907235                    1.077530   
outputsVal   pearson                    0.368961                    0.031818   
             spearman                   0.370699                    0.051911   
             MSE                        1.262078                    1.936364   
             RMSE                       1.121185                    1.389827   
             MAE                        0.891640                    1.073533   

                       situation-dependent_vs_explicit  \
outputsTrain pearson                          0.166586   
             spearman                         0.255461   
             MSE                              1.666828   
             RMSE                             1.289287   
             MAE                              0.983149   
outputsTest  pearson                          0.150458   
             spearman                         0.229466   
             MSE                              1.699083   
             RMSE                             1.301370   
             MAE                              0.990948   
outputsVal   pearson                          0.128679   
             spearman                         0.230083   
             MSE                              1.742641   
             RMSE                             1.318129   
             MAE                              0.995565   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTrain pearson                       0.163371                  0.094793   
             spearman                      0.183950                  0.164308   
             MSE                           1.673258                  1.810413   
             RMSE                          1.291946                  1.343379   
             MAE                           0.977907                  0.999873   
outputsTest  pearson                       0.148287                  0.097105   
             spearman                      0.163899                  0.145859   
             MSE                           1.703427                  1.805790   
             RMSE                          1.303579                  1.340887   
             MAE                           0.990063                  1.002133   
outputsVal   pearson                       0.135723                  0.086190   
             spearman                      0.149576                  0.170089   
             MSE                           1.728554                  1.827620   
             RMSE                          1.312924                  1.349607   
             MAE                           0.994169                  0.995813   

                       compressed_vs_elaborated  
outputsTrain pearson                  -0.072951  
             spearman                 -0.145418  
             MSE                       2.145902  
             RMSE                      1.463513  
             MAE                       1.103398  
outputsTest  pearson                  -0.078206  
             spearman                 -0.171852  
             MSE                       2.156411  
             RMSE                      1.466775  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.900952,1.070497,0.989887,0.987380,0.999273,1.099639
MSE,1.278944,1.939864,1.702851,1.701746,1.814608,2.131629
RMSE,1.128979,1.390984,1.302929,1.302816,1.344624,1.458476
pearson,0.360528,0.030068,0.148575,0.149127,0.092696,-0.065815
spearman,0.360575,0.056258,0.238337,0.165808,0.160086,-0.147081
